# CPHY HDF5 Acquisition Generator + Dask Explorer

Generate many **OpenTelemetry-framed** dense acquisition windows as HDF5 for CPHY
telemetry, land them on RustFS (`s3://cyberphy/…`), and explore
with **Dask + Holoviews/Datashader** out-of-core.

| Layer | Content |
|-------|---------|
| **A — Dataset plane** | Many contiguous `.h5` files under hive prefixes |
| **B — Metadata plane** | Pointer-table *preview* only (no Iceberg write yet) |

**Domain:** cyber-physical (**CPHY**) operations — dense sensor-chain telemetry that
Flink will eventually project into rich OTel spans/metrics. Group tree stays
`ResourceMetrics` for OTel clarity.

**Iceberg File Format API readiness:** fixed one-window files, time bounds in three
places (filename / group attrs / dataset attrs), uuid chain, contiguous Values for
single byte-range kerchunk refs. See
`docs/current/src/architecture/hdf5-iceberg-metadata-plane.md`.

**Profiles**

| Profile | Geometry | Approx Values size |
|---------|----------|--------------------|
| `lab` (default) | 5001 × 10000 × 108 parts | **~10 GiB** (RAID) |
| `lab_tiny` | 256 × 500 × 24 parts | ~6 MiB smoke |
| `airgap_2tb` | 5001 × 10000 × 21500 parts | **~2 TiB** |

Copy this notebook to your home before editing:
```python
import shutil; shutil.copy('/app/sample-notebooks/HDF5_CPHY_Acquisition_Generator.ipynb', '/root/')
```


In [ ]:
# --- User Configuration ---
import os
from datetime import datetime, timezone

# Profile: "lab" | "airgap_2tb"
PROFILE = os.getenv("HDF5_PROFILE", "lab")

# Time unit for Timestamps dataset: "ns" (OTel default) or "us" (microseconds)
TIME_UNIT = os.getenv("HDF5_TIME_UNIT", "ns")

# Contiguous storage (Iceberg/kerchunk-friendly). Set False only for chunk experiments.
CONTIGUOUS = True

# Optional overrides (None = use profile defaults)
N_PARTS = None          # e.g. 8 for a quicker lab run
N_SERIES = None
N_TIME = None

# S3 (JupyterHub injects these for zarf:local)
BUCKET = os.getenv("S3_BUCKET", "cyberphy")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://127.0.0.1:9010")
S3_REGION = os.getenv("AWS_REGION", os.getenv("S3_REGION", "us-east-1"))
# Root is fixed hive layout under the bucket; OUT is the s3:// URL for generate_parts
OUT = f"s3://{BUCKET}/"  # keys always datasets/hdf5/otelcphy/... when hive_layout=True

# Dask
DASK_SCHEDULER = os.getenv(
    "DASK_SCHEDULER_ADDRESS",
    os.getenv("DASK_SCHEDULER", "tcp://cybersec-dask-scheduler.dask.svc.cluster.local:8786"),
)
USE_DASK = os.getenv("USE_DASK", "1") not in ("0", "false", "False")

# Visualization
HEATMAP_SERIES_STRIDE = 1   # plot every Nth series (lab can use 1)
VIZ_MAX_POINTS = 2_000_000  # safety cap for client-side raster before datashader

print(f"PROFILE={PROFILE}  TIME_UNIT={TIME_UNIT}  BUCKET={BUCKET}")
print(f"S3_ENDPOINT={S3_ENDPOINT or '(AWS default)'}")
print(f"DASK={DASK_SCHEDULER if USE_DASK else 'disabled'}")


In [ ]:
# Imports + path to generate_hdf5
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve generate_hdf5.py (repo checkout, image /app, or notebook cwd)
_CANDIDATES = [
    Path("/app/generate_hdf5.py"),              # image-baked (air-gap; survives ConfigMap)
    Path("/app/lib/generate_hdf5.py"),
    Path("/app/sample-notebooks/generate_hdf5.py"),  # ConfigMap or image
    Path.cwd() / "generate_hdf5.py",
    Path.cwd() / "zarf" / "scripts" / "generate_hdf5.py",
    Path("/app/zarf/scripts/generate_hdf5.py"),
    Path("/root/generate_hdf5.py"),
]
_SCRIPT = next((p for p in _CANDIDATES if p.is_file()), None)
if _SCRIPT is None:
    # Fall back: search upward
    for parent in [Path.cwd(), *Path.cwd().parents]:
        cand = parent / "zarf" / "scripts" / "generate_hdf5.py"
        if cand.is_file():
            _SCRIPT = cand
            break
if _SCRIPT is None:
    raise FileNotFoundError("generate_hdf5.py not found — mount repo or copy script into image")

sys.path.insert(0, str(_SCRIPT.parent))
import generate_hdf5 as gh

print(f"Using generator: {_SCRIPT}")
print(f"Profiles: {list(gh.PROFILES)}")
for name, p in gh.PROFILES.items():
    print(f"  {name}: {p['description']}")


In [ ]:
# Build GenConfig
overrides = {
    "time_unit": TIME_UNIT,
    "contiguous": CONTIGUOUS,
    "group_style": "underscore",
    "hive_layout": True,
    "service_name": "cphy-collector",
    "service_namespace": "cyberphy.otel",
    "base_time": datetime.now(timezone.utc).replace(microsecond=0),
}
if N_PARTS is not None:
    overrides["n_parts"] = int(N_PARTS)
if N_SERIES is not None:
    overrides["n_series"] = int(N_SERIES)
if N_TIME is not None:
    overrides["n_time"] = int(N_TIME)

if PROFILE == "custom":
    cfg = gh.GenConfig(**overrides)
else:
    cfg = gh.profile_config(PROFILE, **overrides)

print("=" * 60)
print(f"Active profile: {PROFILE}")
print(f"  series × time × parts = {cfg.n_series} × {cfg.n_time} × {cfg.n_parts}")
print(f"  Values payload ≈ {cfg.estimated_values_bytes()/1e6:.2f} MB "
      f"({cfg.estimated_values_tib():.4f} TiB)")
print(f"  layout={('contiguous' if cfg.contiguous else 'chunked')}  time_unit={cfg.time_unit}")
print(f"  hive keys under s3://{BUCKET}/datasets/hdf5/{cfg.product}/…")
print("=" * 60)
if PROFILE == "lab":
    print("ℹ lab profile is ~10 GiB Values — ensure RUSTFS_DATA_DIR is on /raid.")
if PROFILE == "airgap_2tb":
    print("⚠ airgap_2tb is ~2 TiB — only run on a large air-gap cluster with ample S3.")
    print("  For this notebook session, consider N_PARTS=2 smoke test first.")


## Iceberg-oriented layout (dataset plane)

```
s3://cyberphy/
  datasets/hdf5/otelcphy/                 # Layer A — conserved, never rewritten by Iceberg
    service=cphy-collector/
      date=YYYY-MM-DD/
        hour=HH/
          otelcphy_<YYYYMMDDTHHMMSS>_<part>Z.h5
  indexes/kerchunk/<fingerprint>.json     # optional refs (Layer B stub)
  # iceberg/warehouse/                    # Layer B warehouse — NOT written by this notebook
```

**Why many small files?** Each file is one fixed-duration acquisition window. That maps
cleanly to Iceberg *data files* under a future HDF5 `FormatModel` (File Format API), and
to pointer-table rows (`telemetry.hdf5_datasets`) with `t_min_ns`/`t_max_ns` pruning today.

**Contiguous Values:** one byte range per dataset → kerchunk refs degenerate to a single
`(offset, length)`; series hyperslabs are arithmetic strides (`n_time * itemsize` per row).


In [ ]:
# Generate parts → RustFS
inventory = gh.generate_parts(
    cfg,
    OUT,
    s3_endpoint=S3_ENDPOINT or None,
    emit_kerchunk_refs=False,
    dry_run=False,
    progress_every=max(1, cfg.n_parts // 8),
)

pointer_rows = gh.inventory_to_pointer_rows(inventory)
inv_df = pd.DataFrame(pointer_rows)
print(inv_df.head())
print(f"\nTotal files: {len(inv_df)}  total size: {inv_df['size_bytes'].sum()/1e6:.1f} MB")


In [ ]:
# Upload pointer-table preview (not full Iceberg — just inventory for later registration)
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as pafs

def s3_filesystem():
    kw = dict(
        access_key=os.environ.get("AWS_ACCESS_KEY_ID", "admin"),
        secret_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "admin"),
        region=S3_REGION,
    )
    token = os.environ.get("AWS_SESSION_TOKEN") or ""
    if token:
        kw["session_token"] = token
    if S3_ENDPOINT:
        kw["endpoint_override"] = S3_ENDPOINT
        kw["scheme"] = "https" if S3_ENDPOINT.startswith("https") else "http"
    return pafs.S3FileSystem(**kw)

s3fs_pa = s3_filesystem()
inv_key = f"{BUCKET}/datasets/hdf5/{cfg.product}/_inventory/parts.parquet"
table = pa.Table.from_pandas(inv_df)
pq.write_table(table, inv_key, filesystem=s3fs_pa)
print(f"Wrote pointer-table preview → s3://{inv_key}")

# s3fs listing of first hive prefix
import s3fs
s3 = s3fs.S3FileSystem(
    key=os.environ.get("AWS_ACCESS_KEY_ID", "admin"),
    secret=os.environ.get("AWS_SECRET_ACCESS_KEY", "admin"),
    client_kwargs={"endpoint_url": S3_ENDPOINT} if S3_ENDPOINT else {},
    config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
)
prefix = f"{BUCKET}/datasets/hdf5/{cfg.product}"
print("Sample listing:")
for p in sorted(s3.glob(prefix + "/**/*.h5"))[:8]:
    print(" ", p)


## Structural audit (one file)

Open a single acquisition and verify: contiguous Values, uuid chain, time triple agreement,
and Metric sub-window vs acquisition start index.


In [ ]:
import h5py
import tempfile

# Download one part for audit (h5py needs a seekable path for full attr walk)
sample_uri = inventory[0]["uri"]
sample_key = inventory[0]["key"]
print("Auditing", sample_uri)

with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
    s3.get(f"{BUCKET}/{sample_key}", tmp.name)
    with h5py.File(tmp.name, "r") as f:
        def walk(name, obj):
            kind = "G" if isinstance(obj, h5py.Group) else "D"
            nattr = len(obj.attrs)
            extra = ""
            if isinstance(obj, h5py.Dataset):
                extra = f" shape={obj.shape} dtype={obj.dtype} contiguous={obj.chunks is None}"
            print(f"  {kind} /{name}  attrs={nattr}{extra}")
        print("Tree:")
        f.visititems(walk)
        print("\nRoot uuid:", f.attrs.get("collection.uuid"))
        rm = f["ResourceMetrics"]
        print("Acquisition start.series.index:", int(rm.attrs["start.series.index"]))
        met_name = "Metric_0" if "Metric_0" in rm else "Metric[0]"
        met = rm[met_name]
        print("Metric start.series.index:", int(met.attrs["start.series.index"]))
        print("  (sub-window ≠ acquisition — expected)")
        vals = met["Values"]
        ts = met["Timestamps"]
        print(f"Values: {vals.shape} chunks={vals.chunks} scale.factor={float(met.attrs.get('scale.factor', 0))}")
        print(f"Timestamps: unit={ts.attrs.get('unit')} start.index={int(ts.attrs['start.index'])}")
        print(f"part.start.time Values:", vals.attrs.get("part.start.time"))
        print(f"part.start.time RM:    ", rm.attrs.get("start.time"))


## Dask client

Workers inherit S3 env from the DaskCluster (admin/cyberphy on lab). Tasks should only
load **hyperslabs**, never whole multi-hundred-MB files into the client.


In [ ]:
from dask.distributed import Client, get_client

client = None
if USE_DASK:
    try:
        client = Client(DASK_SCHEDULER, timeout="10s")
        print("Connected:", client)
        print("Dashboard:", client.dashboard_link)
        print("Workers:", len(client.scheduler_info().get("workers", {})))
    except Exception as e:
        print(f"Dask unavailable ({e}); falling back to local threads")
        client = Client(processes=False, threads_per_worker=2, n_workers=2)
        print(client)
else:
    print("USE_DASK=0 — client-local only")


In [ ]:
# Dask-ready hyperslab reader (one part → numpy block)
# Paths on workers: use s3fs inside the task so workers need network to RustFS.

def read_values_hyperslab(bucket, key, endpoint, access_key, secret_key,
                          series_slice=None, time_slice=None):
    """Read Values[series, time] from one HDF5 object on S3."""
    import tempfile
    import h5py
    import numpy as np
    import s3fs

    s3 = s3fs.S3FileSystem(
        key=access_key, secret=secret_key,
        client_kwargs={"endpoint_url": endpoint} if endpoint else {},
        config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
    )
    with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
        s3.get(f"{bucket}/{key}", tmp.name)
        with h5py.File(tmp.name, "r") as f:
            met = f["ResourceMetrics"]["Metric_0"] if "Metric_0" in f["ResourceMetrics"] \
                  else f["ResourceMetrics"]["Metric[0]"]
            ds = met["Values"]
            ss = series_slice or slice(None)
            ts = time_slice or slice(None)
            arr = np.asarray(ds[ss, ts], dtype=np.float32)
            scale = float(met.attrs.get("scale.factor", 1.0)) or 1.0
            # denormalize toward physical-ish units for viz (phase-like)
            return arr / scale


AK = os.environ.get("AWS_ACCESS_KEY_ID", "admin")
SK = os.environ.get("AWS_SECRET_ACCESS_KEY", "admin")

# Build delayed stack of parts (series × time) for out-of-core access
import dask.array as da
from dask import delayed

keys = [m["key"] for m in inventory]
# For lab, load full series; for larger profiles, stride series on read
s_slice = slice(None)
t_slice = slice(None)

delayed_reads = [
    delayed(read_values_hyperslab)(BUCKET, k, S3_ENDPOINT, AK, SK, s_slice, t_slice)
    for k in keys
]
# Each part → (n_series, n_time); stack along a new "part" axis then reshape to time
sample = read_values_hyperslab(BUCKET, keys[0], S3_ENDPOINT, AK, SK, s_slice, t_slice)
print("Sample part shape:", sample.shape, sample.dtype)

blocks = [
    da.from_delayed(d, shape=sample.shape, dtype=np.float32)
    for d in delayed_reads
]
# Concatenate along time: (series, time_total)
values_da = da.concatenate(blocks, axis=1)
print("Dask array:", values_da)
print("npartitions (blocks):", values_da.npartitions)


## Holoviews + Datashader

Datashader aggregates on the **cluster or client** without shipping every point to the
browser. For the lab profile we pull a modest compute(); for airgap-scale, keep work on
Dask and only materialize decimated overviews.


In [ ]:
import holoviews as hv
from holoviews.operation.datashader import datashade, rasterize, dynspread
import datashader as ds
hv.extension("bokeh")

# Materialize for viz — lab is small; for large profiles, decimate first
max_series_plot = min(cfg.n_series, 512)
max_time_plot = min(values_da.shape[1], 4000)

viz = values_da[:max_series_plot, :max_time_plot]
print("Computing viz slab", viz.shape, "…")
viz_np = viz.compute()
print("done", viz_np.shape, "minmax", float(viz_np.min()), float(viz_np.max()))

# Build a points table: time index × series index × value  (sampled if huge)
n_s, n_t = viz_np.shape
# stride so total points stay under VIZ_MAX_POINTS
stride_s = max(1, int(np.ceil(n_s * n_t / VIZ_MAX_POINTS)))
stride_t = 1
sub = viz_np[::stride_s, ::stride_t]
ss, tt = sub.shape
# coordinates
series_idx = np.arange(0, n_s, stride_s)[:ss]
time_idx = np.arange(0, n_t, stride_t)[:tt]
# mesh to columns
T, S = np.meshgrid(time_idx, series_idx)
df = pd.DataFrame({
    "time_sample": T.ravel().astype(np.float32),
    "series": S.ravel().astype(np.float32),
    "value": sub.ravel().astype(np.float32),
})
print(f"Datashader points: {len(df):,} (stride_s={stride_s})")

points = hv.Points(df, kdims=["time_sample", "series"], vdims=["value"])
heatmap = datashade(points, aggregator=ds.mean("value"), cmap="fire").opts(
    width=900,
    height=420,
    title=f"CPHY acquisition phase (mean) — {PROFILE} — {len(keys)} parts",
    xlabel="time sample (concatenated parts)",
    ylabel="series index",
)
dynspread(heatmap, threshold=0.4, max_px=3)


In [ ]:
# Series profile + value histogram (client-side on viz_np)
mid = viz_np.shape[0] // 2
profile = hv.Curve(
    (np.arange(viz_np.shape[1]), viz_np[mid]),
    kdims="time_sample", vdims="value",
).opts(width=900, height=200, title=f"Series {mid} profile", color="#4fc3f7")

hist = hv.Histogram(np.histogram(viz_np.ravel()[::10], bins=80)).opts(
    width=900, height=200, title="Value distribution (decimated)",
    xlabel="normalized phase",
)
(profile + hist).cols(1)


In [ ]:
# Multi-file time continuity (Timestamps stitch check)
def read_ts_bounds(bucket, key, endpoint, access_key, secret_key):
    import tempfile, h5py, s3fs
    s3 = s3fs.S3FileSystem(
        key=access_key, secret=secret_key,
        client_kwargs={"endpoint_url": endpoint} if endpoint else {},
        config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
    )
    with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
        s3.get(f"{bucket}/{key}", tmp.name)
        with h5py.File(tmp.name, "r") as f:
            met = f["ResourceMetrics"]["Metric_0"] if "Metric_0" in f["ResourceMetrics"] \
                  else f["ResourceMetrics"]["Metric[0]"]
            ts = met["Timestamps"]
            return int(ts[0]), int(ts[-1]), int(ts.attrs["start.index"]), str(ts.attrs.get("unit", b"ns"))

bounds = []
for m in inventory[: min(12, len(inventory))]:
    a, b, si, unit = read_ts_bounds(BUCKET, m["key"], S3_ENDPOINT, AK, SK)
    bounds.append({"part": m["part_idx"], "t0": a, "t1": b, "start.index": si, "unit": unit})
bdf = pd.DataFrame(bounds)
print(bdf)
# gaps between parts (should be ~ one sample step)
if len(bdf) > 1:
    gaps = bdf["t0"].values[1:] - bdf["t1"].values[:-1]
    print("inter-part Δ (should be ~step):", gaps[:5], "…")


## Next steps (metadata plane)

1. **Registration pipeline** — scan hive prefixes → append `telemetry.hdf5_datasets` +
   `hdf5_chunk_stats` (pointer table / virtual chunks).
2. **Kerchunk** — `--emit-kerchunk` or notebook cell; store under `indexes/kerchunk/`.
3. **Overviews** — Iceberg parquet pyramids for first-paint datashader (see design §6).
4. **File Format API** — when an HDF5 `FormatModel` exists, these files become first-class
   Iceberg data files; pointer table remains provenance.

### Air-gap 2 TiB checklist

```python
PROFILE = "airgap_2tb"
# or: PROFILE="lab"; N_PARTS=2  # smoke
```

Ensure RustFS/S3 capacity ≥ ~2.2 TiB free, Dask workers with spill, and run generation as a
batch job (not a long-lived notebook cell) if part count is full 21500.


In [ ]:
# Optional cleanup of client
# if client is not None:
#     client.close()
print("Notebook complete.")
print(f"Files: {len(inventory)} under s3://{BUCKET}/datasets/hdf5/{cfg.product}/")
